In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True) 

import jax.numpy as jnp
import numpy as np
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0
from src.devices.qutip_device import HiddenDevice
from src.analysis.metrics import bell_fid, leakage
from src.analysis.plots import plot_absolute_cityscape
from src.control.waveforms import (
    build_local_drive_operators,
    generate_bell_state,
)
from src.calibration.virtual_z_cal import calibrate_virtual_z

# Load Configurations
config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
dt = config.pulse.dt_ns

# Build static JAX bases
H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=config.hardware.levels)
# Build the 9D single-qubit drive operators
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=config.hardware.levels)

# Load the optimized CZ pulse from the previous calibration step
cal_pulse_rad = np.load(config.artifact_path("cal_pulse_rad.npy"))
print(f"Loaded optimized CZ pulse from {config.artifact_path('cal_pulse_rad.npy')}")

cal_g, cal_d, cal_a = np.load(config.artifact_path("cal_params.npy"))

H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)


In [ ]:

import matplotlib.pyplot as plt

duration_ns = 20.0

# Calibrate Virtual-Z phases on the Digital Twin (propagators computed once)
vz_cal = calibrate_virtual_z(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad, 
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control
)
c1, c2 = vz_cal["v_phase1"], vz_cal["v_phase2"]


print(f"Digital Twin Calibration -> Best Infidelity: {vz_cal['infidelity']*100:.2f}% (Ceiling Fidelity: {vz_cal['fidelity']*100:.2f}%)")
print(f"Calibrated Phases -> Q1 (tomography frame): {c1:5.2f} rad | Q2 (U3 drive frame): {c2:5.2f} rad\n")

# Plot the 2D Virtual-Z Calibration Landscape
plt.figure(figsize=(6, 4.5))
plt.contourf(
    vz_cal["grid_rad"], vz_cal["grid_rad"], vz_cal["fidelity_grid"].T, 
    levels=25, cmap="viridis"
)
plt.colorbar(label=r"$\Phi^+$ State Overlap")
plt.plot(c1, c2, "x", markersize=6, label=f"Optimum ({c1:.2f}, {c2:.2f} rad)")
plt.xlabel("Virtual-Z Phase Q1 (rad)")
plt.ylabel("Virtual-Z Phase Q2 (rad)")
plt.title("Digital Twin Virtual-Z Calibration Landscape")
plt.legend()
plt.tight_layout()
plt.show()

# Execute single-shot sequence on the Digital Twin (noiseless ceiling)
rho_corr = generate_bell_state(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad, 
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control,
    v_phase1=c1, v_phase2=c2
)

populations = np.abs(np.real(np.diag(rho_corr)))
print(f"Digital Twin Density Matrix (|rho|):\n{np.abs(rho_corr)}")
print(f"Populations: |00>: {populations[0]:.3f} | |01>: {populations[1]:.3f} | |10>: {populations[2]:.3f} | |11>: {populations[3]:.3f}")

fig = plot_absolute_cityscape(rho_corr)
plt.show()

In [ ]:
rho_phys_qobj = qt.Qobj(rho_corr)
rho_mat = rho_phys_qobj.full()

# Mask phases where magnitude is near zero (< 1e-6) to avoid meaningless angle noise
masked_phases = np.where(np.abs(rho_mat) >= 1e-6, np.angle(rho_mat), 0.0)

print(f"Phases (rad, masked for |rho| < 1e-6):\n{np.round(masked_phases, 3)}\n")
print(f"Absolute values:\n{np.round(np.abs(rho_mat), 3)}\n")

fidelity_twin = bell_fid(rho_corr)
print(f"Digital Twin Bell State Fidelity (Noiseless Ceiling): {fidelity_twin * 100:.2f}%")


In [ ]:
from src.control.virtual_z import apply_frame
from src.control.waveforms import pack_sequence, build_bell_waveforms


idx = [0, 1, 3, 4]
rho00 = qt.ket2dm(qt.tensor(qt.basis(3, 0), qt.basis(3, 0)))
fridge = HiddenDevice.from_config(config)
assert abs(fridge.dt - dt) < 1e-12

wf = build_bell_waveforms(duration_ns, dt, cal_a, cal_d, cal_pulse_rad, v_phase2=c2)

rho_f = fridge.play_sequence(pack_sequence(wf), rho00)
rho_framed = apply_frame(rho_f, c1, c2)
F_B = bell_fid(rho_framed.full())
leakage_f = leakage(rho_f.full())

# Population breakdown, to see where the loss is
p = np.real(np.diag(rho_f.full()))
print(f"Twin vs device Bell fidelity: {fidelity_twin*100:.2f}% vs {F_B*100:.2f}% (difference: {(fidelity_twin - F_B)*100:.2f}%)")
print(f"   P(comp. subspace) = {p[idx].sum():.4f}  (leakage: {leakage_f*100:.2f}%)")

np.savez(config.artifact_path("bell_run.npz"), rho_f=rho_f.full(), v_phase1=c1, v_phase2=c2)
